# Second 1.0 — Phase 5: Supervised Fine-Tuning (SFT) on 2B

This notebook trains and evaluates the **Second-1.0-2B-SFT** model on Google Colab with an **NVIDIA L4 GPU**.
It follows the Phase 5 specification from `PLAN.md`.

### Phase 5 Specification

| Setting | Value |
|---|---|
| **Base Model** | `Qwen/Qwen3.5-2B` (Instruct, text-only, non-thinking) |
| **Fine-Tuning Method** | LoRA (rank 32, alpha 32, dropout 0.05, all text linear layers) |
| **Training Data** | Filtered human SFT dataset (`data/processed/sft_train.jsonl`, ~5,522 examples) |
| **Validation Data** | SFT validation split (`data/processed/sft_val.jsonl`, 137 examples) |
| **Epochs** | 2 |
| **Batch Size** | 4 per device, 2 gradient accumulation steps (effective batch size: 8) |
| **Learning Rate** | 1e-4, cosine decay, 3% warmup ratio |
| **Max Sequence Length** | 1,024 tokens |
| **Loss Masking** | Answer tokens only (prompt tokens masked with -100) |
| **Checkpoint Strategy** | Save to Google Drive every 200 steps (max 3 checkpoints) |
| **Pass Rule** | Format pass rate > 95.0% and hard-fail rate < base Qwen 2B |

### Step-by-Step Instructions

1. **Step 1: Verify GPU Hardware**: Check that an NVIDIA L4 GPU with 24 GB VRAM is active.
2. **Step 2: Mount Google Drive**: Mount persistent storage for checkpoints and dataset backups.
3. **Step 3: Load Phase 5 Data and Code Bundle**: Load and extract `phase5_colab_bundle.zip`.
4. **Step 4: Install Dependencies**: Install pinned packages and fast attention kernels.
5. **Step 5: Load Qwen 3.5 2B in Non-Thinking Mode with LoRA**: Prepare model in `torch.bfloat16`.
6. **Step 6: Prepare SFT Datasets with Answer-Only Loss Masking**: Tokenize and mask prompts.
7. **Step 7: Run SFT Training**: Train for 2 epochs with checkpoint saving and auto-resume.
8. **Step 8: Generate Greedy Outputs**: Produce completions on validation and test sets.
9. **Step 9: Run Phase 4 Metrics and Export Results**: Verify release rule and download results bundle.


In [ ]:
# Step 1: Verify GPU Hardware
!nvidia-smi

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU Name: {device_name}")
print(f"Total VRAM: {vram_gb:.2f} GB")
print(f"bfloat16 supported: {bf16_supported}")

if "L4" not in device_name:
    print(f"Notice: Phase 5 targets an NVIDIA L4 GPU. Detected {device_name}.")


In [ ]:
# Step 2: Mount Google Drive for persistent checkpoint storage
import os
from pathlib import Path

CHECKPOINT_DIR = "/content/drive/MyDrive/second/checkpoints/sft-1.0-2b"
DATA_DRIVE_DIR = "/content/drive/MyDrive/second/data"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
    print("Google Drive mounted successfully.")
    print(f"Checkpoint directory: {CHECKPOINT_DIR}")
    print(f"Data directory: {DATA_DRIVE_DIR}")
except Exception as exc:
    print(f"Drive mount notice: {exc}")
    CHECKPOINT_DIR = "/content/second_checkpoints/sft-1.0-2b"
    DATA_DRIVE_DIR = "/content/second_data"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
    print(f"Using local fallback checkpoint directory: {CHECKPOINT_DIR}")


In [ ]:
# Step 3: Load Phase 5 Data and Code Bundle
import os
import shutil
import sys
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/second_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    Path("/content/drive/MyDrive/second/data/phase5_colab_bundle.zip"),
    Path("/content/phase5_colab_bundle.zip"),
    Path("/content/data/processed/phase5_colab_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path is None:
    if Path("eval/testset-1.0.jsonl").is_file() and Path("data/processed/sft_train.jsonl").is_file():
        print("Detected local repository checkout. Creating bundle...")
        from training.build_colab_bundle import build_colab_bundle
        bundle_path = build_colab_bundle()
    else:
        print("phase5_colab_bundle.zip not found on Drive or /content.")
        print("Please upload phase5_colab_bundle.zip using the file picker:")
        try:
            from google.colab import files
            uploaded = files.upload()
            for filename in uploaded.keys():
                if filename.endswith(".zip"):
                    uploaded_path = Path("/content") / filename
                    drive_copy = Path("/content/drive/MyDrive/second/data") / filename
                    if drive_copy.parent.is_dir():
                        shutil.copyfile(uploaded_path, drive_copy)
                        print(f"Saved copy to Google Drive: {drive_copy}")
                    bundle_path = uploaded_path
                    break
        except Exception as exc:
            print(f"Colab upload notice: {exc}")

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print("Bundle extracted successfully.")
else:
    raise FileNotFoundError(
        "Could not find or upload phase5_colab_bundle.zip. "
        "Run python training/build_colab_bundle.py locally and upload the zip."
    )

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
    print(f"Added {WORK_DIR} to sys.path.")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# Step 4: Install Dependencies
!pip install -q --upgrade pip
!pip uninstall -y -q torchao torchvision 2>/dev/null || true
!pip install -q "transformers>=4.49.0" "trl>=0.15.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.0.0" "bitsandbytes>=0.45.0" "chess" "pyyaml" "langdetect"

# Optional fast attention kernels (prebuilt wheels only to avoid compilation delay)
print("Installing optional fast kernels (prebuilt binaries only)...")
!pip install -q causal-conv1d --only-binary :all: 2>/dev/null || echo "causal-conv1d prebuilt binary not found. PyTorch fallback active."
!pip install -q flash-linear-attention --only-binary :all: 2>/dev/null || echo "flash-linear-attention prebuilt binary not found. PyTorch fallback active."

print("Dependencies installed successfully.")


In [ ]:
# Step 5: Load Qwen 3.5 2B Instruct in Non-Thinking Text-Only Mode with LoRA
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import peft.import_utils
import peft.tuners.lora.torchao
from training.sft_data import load_sft_config, resolve_text_linear_modules

cfg = load_sft_config("training/configs/second-1.0-2b.yaml")

# Bypass incompatible preinstalled torchao version in Google Colab
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False

MODEL_ID = cfg.base_model
print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Verify chat template with enable_thinking=False
test_msg = [
    {"role": "system", "content": "You are Second, a chess assistant."},
    {"role": "user", "content": "Explain 1. e4."}
]
try:
    formatted_prompt = tokenizer.apply_chat_template(
        test_msg,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    print("Tokenizer supports enable_thinking=False.")
except TypeError:
    formatted_prompt = tokenizer.apply_chat_template(
        test_msg,
        tokenize=False,
        add_generation_prompt=True,
    )
    print("Tokenizer formatted template without enable_thinking.")

print(f"Sample formatted prompt header:\n{formatted_prompt[:180]}...\n")

print(f"Loading base model {MODEL_ID} in bfloat16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

# Identify all text linear modules (excludes vision towers and lm_head)
target_modules = resolve_text_linear_modules(model.named_modules())
print(f"Identified LoRA target modules: {target_modules}")

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=target_modules,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Prepare SFT Datasets with Answer-Only Loss Masking
from datasets import Dataset
from training.sft_data import (
    AnswerOnlyDataCollator,
    load_sft_config,
    load_sft_records,
    tokenize_sft_record,
)

cfg = load_sft_config("training/configs/second-1.0-2b.yaml")
TRAIN_DATA_PATH = cfg.train_data_path
VAL_DATA_PATH = cfg.val_data_path

train_records = load_sft_records(TRAIN_DATA_PATH, max_examples=cfg.max_examples)
val_records = load_sft_records(VAL_DATA_PATH, max_examples=cfg.max_examples)

print(f"Loaded {len(train_records)} train records from {TRAIN_DATA_PATH}.")
print(f"Loaded {len(val_records)} val records from {VAL_DATA_PATH}.")

def encode_text(text: str) -> list[int]:
    return tokenizer.encode(text, add_special_tokens=False)

def tokenize_record_map(rec):
    return tokenize_sft_record(
        rec,
        encode_fn=encode_text,
        max_seq_length=cfg.max_seq_length,
    )

train_tokenized = [tokenize_record_map(r) for r in train_records]
val_tokenized = [tokenize_record_map(r) for r in val_records]

train_dataset = Dataset.from_list(train_tokenized)
val_dataset = Dataset.from_list(val_tokenized)

# Verify answer-only loss masking on sample 0
sample_feat = train_tokenized[0]
prompt_token_count = sample_feat["labels"].count(-100)
active_token_count = len(sample_feat["labels"]) - prompt_token_count

print("Verification sample 0:")
print(f"- Total sequence length: {len(sample_feat['input_ids'])} tokens")
print(f"- Masked prompt tokens (loss = -100): {prompt_token_count} tokens")
print(f"- Active answer tokens (loss computed): {active_token_count} tokens")

assert prompt_token_count > 0, "Prompt tokens must be masked with -100."
assert active_token_count > 0, "Answer tokens must not be masked."
assert all(lbl == -100 for lbl in sample_feat["labels"][:prompt_token_count]), "Prompt tokens must all be -100."

data_collator = AnswerOnlyDataCollator(
    pad_token_id=tokenizer.pad_token_id or 0,
    label_pad_token_id=-100,
    return_tensors="pt",
)
print("Datasets and answer-only collator prepared successfully.")

In [ ]:
# Step 7: Run SFT Training with Checkpoint Saving Every 200 Steps
import glob
import json
import time
from transformers import Trainer, TrainingArguments
from training.sft_data import load_sft_config

cfg = load_sft_config("training/configs/second-1.0-2b.yaml")

# Look for existing checkpoints to support automatic resumption
existing_ckpts = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
latest_checkpoint = existing_ckpts[-1] if existing_ckpts else None
if latest_checkpoint:
    print(f"Found existing checkpoint. Resuming from: {latest_checkpoint}")
else:
    print(f"Starting fresh training into: {CHECKPOINT_DIR}")

total_train_steps = max(1, (len(train_dataset) // (cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps)) * cfg.num_train_epochs)
warmup_steps = max(1, int(cfg.warmup_ratio * total_train_steps))

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler_type,
    warmup_steps=warmup_steps,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    save_steps=cfg.save_steps,
    save_total_limit=3,
    eval_strategy=cfg.eval_strategy,
    eval_steps=cfg.eval_steps,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

start_time = time.time()
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
training_time = time.time() - start_time

print(f"Training completed in {training_time / 60.0:.2f} minutes.")
trainer.save_metrics("train", train_result.metrics)

# Save final adapter and tokenizer
final_adapter_dir = os.path.join(CHECKPOINT_DIR, "final_adapter")
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Saved final adapter and tokenizer to: {final_adapter_dir}")

training_summary = {
    "model_id": MODEL_ID,
    "epochs": cfg.num_train_epochs,
    "global_steps": trainer.state.global_step,
    "train_loss": float(train_result.training_loss),
    "training_time_minutes": round(training_time / 60.0, 2),
    "final_adapter_path": final_adapter_dir,
}
summary_path = os.path.join(CHECKPOINT_DIR, "sft_training_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)
print(f"Saved training summary to {summary_path}")

In [ ]:
# Step 8: Generate Greedy Outputs on Validation Split and Frozen Test Set
import json
from pathlib import Path
from eval.baselines import load_jsonl
from eval.metrics import clean_completion_text
from training.sft_data import build_non_thinking_chat_prompt

def run_batched_generation(
    records: list[dict],
    model,
    tokenizer,
    batch_size: int = 16,
    max_new_tokens: int = 256,
) -> list[dict]:
    """Generate greedy completions for prompt records with batched inference."""
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    stop_ids = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(int(tokenizer.eos_token_id))
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    model.eval()
    results = []

    for i in range(0, len(records), batch_size):
        batch = records[i:i + batch_size]
        prompts = [
            r.get("chat_prompt") or build_non_thinking_chat_prompt(r["user_message"])
            for r in batch
        ]
        
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        inputs.pop("token_type_ids", None)
        input_len = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=stop_ids if stop_ids else tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        for j, rec in enumerate(batch):
            gen_tokens = outputs[j][input_len:]
            decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            cleaned = clean_completion_text(decoded)

            out_rec = {k: v for k, v in rec.items() if k != "comment"}
            out_rec["completion"] = cleaned
            out_rec["output"] = cleaned
            out_rec["model"] = "Second-1.0-2B-SFT"
            results.append(out_rec)

    tokenizer.padding_side = orig_padding_side
    return results

val_items = load_jsonl("data/processed/sft_val.jsonl")
test_items = load_jsonl("eval/testset-1.0.jsonl")

print(f"Generating greedy outputs for {len(val_items)} validation items...")
val_outputs = run_batched_generation(val_items, model, tokenizer, batch_size=16)

print(f"Generating greedy outputs for {len(test_items)} frozen test set items...")
test_outputs = run_batched_generation(test_items, model, tokenizer, batch_size=16)

# Save outputs to checkpoint directory and local eval results directory
eval_res_dir = Path("eval/results")
eval_res_dir.mkdir(parents=True, exist_ok=True)

val_out_paths = [
    os.path.join(CHECKPOINT_DIR, "sft-1.0-2b_val_outputs.jsonl"),
    "eval/results/sft-1.0-2b_val_outputs.jsonl",
]
test_out_paths = [
    os.path.join(CHECKPOINT_DIR, "sft-1.0-2b_outputs.jsonl"),
    "eval/results/sft-1.0-2b_outputs.jsonl",
]

for p in val_out_paths:
    with open(p, "w", encoding="utf-8") as f:
        for r in val_outputs:
            f.write(json.dumps(r) + "\n")
    print(f"Saved validation outputs: {p}")

for p in test_out_paths:
    with open(p, "w", encoding="utf-8") as f:
        for r in test_outputs:
            f.write(json.dumps(r) + "\n")
    print(f"Saved test outputs: {p}")


In [ ]:
# Step 9: Run Phase 4 Automatic Metrics, Inspect 30 Outputs, and Export Results Bundle
import shutil
import zipfile
from training.evaluate_sft import run_phase5_evaluation

print("Executing Phase 5 evaluation runner...")
summary = run_phase5_evaluation(
    val_outputs_path="eval/results/sft-1.0-2b_val_outputs.jsonl",
    test_outputs_path="eval/results/sft-1.0-2b_outputs.jsonl",
    base_qwen_path="eval/results/baselines/base-qwen-2b.jsonl",
    results_dir="eval/results",
    reports_dir="eval/reports",
    model_name="sft-1.0-2b",
    training_metadata=training_summary if "training_summary" in locals() else None,
)

print("\n" + "=" * 60)
print("PHASE 5 RELEASE RULE VERIFICATION:")
print(f"- Overall Pass Status: {'PASS' if summary['passed_phase5'] else 'FAIL'}")
print(f"- Format Pass Rate: {summary['pass_rule']['format_pass_rate']:.1f}% (Required: > 95.0%)")
print(f"- SFT Hard-Fail Rate: {summary['pass_rule']['sft_hard_fail_rate']:.1f}%")
print(f"- Base Qwen 2B Hard-Fail Rate: {summary['pass_rule']['base_hard_fail_rate']:.1f}%")
print("=" * 60 + "\n")

report_text = Path("eval/reports/sft-1.0-2b.md").read_text(encoding="utf-8")
print(f"Full report written to: eval/reports/sft-1.0-2b.md ({len(report_text)} characters)\n")

# Package results into zip bundle for easy download and Drive sync
results_zip_name = "phase5_sft_results.zip"
files_to_pack = [
    ("eval/results/sft-1.0-2b_val_outputs.jsonl", "sft-1.0-2b_val_outputs.jsonl"),
    ("eval/results/sft-1.0-2b_outputs.jsonl", "sft-1.0-2b_outputs.jsonl"),
    ("eval/results/sft-1.0-2b_val_eval.json", "sft-1.0-2b_val_eval.json"),
    ("eval/results/sft-1.0-2b_eval.json", "sft-1.0-2b_eval.json"),
    ("eval/reports/sft-1.0-2b.md", "sft-1.0-2b.md"),
]
if os.path.exists(os.path.join(CHECKPOINT_DIR, "sft_training_summary.json")):
    files_to_pack.append(
        (os.path.join(CHECKPOINT_DIR, "sft_training_summary.json"), "sft_training_summary.json")
    )

with zipfile.ZipFile(results_zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files_to_pack:
        if os.path.exists(src):
            zf.write(src, arcname=arc)

# Copy to Google Drive
drive_zip_path = os.path.join(CHECKPOINT_DIR, results_zip_name)
shutil.copyfile(results_zip_name, drive_zip_path)
print(f"Saved results zip to Google Drive: {drive_zip_path}")

# Trigger browser download if running in Colab
try:
    from google.colab import files
    files.download(results_zip_name)
    print(f"Triggered browser download for {results_zip_name}.")
except Exception as exc:
    print(f"Browser download notice: {exc}")

# Stop the runtime when a run ends to save compute units
try:
    from google.colab import runtime
    print("Stopping Colab runtime to save compute units...")
    runtime.unassign()
except Exception as exc:
    print(f"Runtime disconnect notice: {exc}")